In [1]:
import os
import pandas as pd
from urllib import request
from astroquery.gaia import Gaia
from kineamtics_and_dynamics import UVW, gal_population, compute_orbit_params

The archive is unstable and may perform below expectations. Please avoid launching intense Python query showers. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the recent infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


In [2]:
"""
Table Access Protocol (TAP)

This is the link to the NEA table of confirmed planets (from "ps" == Planetary Systems) 
with default flag = 1 (default_flag+=+1)
The table is saved in CSV format (format=csv)
"""

download_link = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+ps+where+default_flag+=+1&format=csv'

def _create_data_dir():
    """ Create empty directory where nea_full.csv will be stored """
    current_dir = os.getcwd()
    table_directory = os.path.join(current_dir, 'nea_tables')
    if not os.path.exists(table_directory):
        os.makedirs(table_directory)


def _check_data_dir():
    current_dir = os.getcwd()
    table_directory = os.path.join(current_dir, 'nea_tables')
    return os.path.exists(table_directory)


def get_data_dir():
    """ Return directory where nea_full.csv is stored """
    if not _check_data_dir():
        _create_data_dir()
        
    current_dir = os.getcwd()
    return os.path.join(current_dir, 'nea_tables')


def download_nea_table():
    """ Download and save nea_full.csv table in ./nea_tables directory  """
    local_file = os.path.join(get_data_dir(), 'nea_full.csv')

    with request.urlopen(download_link) as response:
        data = response.read()

    with open(local_file, 'wb') as f:
        f.write(data)

def open_nea_table():
    """ Opens the NEA full table as a pandas dataframe"""
    current_dir = os.getcwd()
    table_directory = os.path.join(current_dir, 'nea_tables', 'nea_full.csv')
    nea_full_table = pd.read_csv(table_directory, low_memory=False)

    return nea_full_table

In [3]:
#nea_full_table = download_nea_table()
nea_full_table = open_nea_table()

In [4]:
nea_gaia_clean = nea_full_table[nea_full_table['gaia_dr3_id'].notna()].copy()
nea_gaia_clean['source_id'] = nea_gaia_clean['gaia_dr3_id'].str.replace('Gaia DR3 ', '', regex=False).str.strip()
nea_gaia_clean = nea_gaia_clean[nea_gaia_clean['source_id'].ne('')]
nea_gaia_clean.shape


(5774, 356)

In [5]:
GAIA_COLUMNS = [
    'source_id', 'ra', 'dec', 'parallax', 'parallax_error',
    'pmra', 'pmdec', 'radial_velocity',
    'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag',
    'mh_gspphot', 'mh_gspspec',
]

KINEMATIC_COLUMNS = [
    'U_lsr', 'V_lsr', 'W_lsr', 'P_gal',
    'rapo', 'rperi', 'eccentricity', 'zmax',
]

def fetch_gaia_sources(source_ids, chunk_size=500, verbose=True):
    unique_ids = pd.Series(source_ids, dtype='string').dropna().astype(str).str.strip()
    unique_ids = unique_ids[unique_ids.ne('')].drop_duplicates().tolist()

    frames = []
    for start in range(0, len(unique_ids), chunk_size):
        batch = unique_ids[start:start + chunk_size]
        query_ids = ', '.join(batch)
        query = f"""
        SELECT gs.source_id, gs.ra, gs.dec, gs.parallax, gs.parallax_error,
               gs.pmra, gs.pmdec, gs.radial_velocity,
               gs.phot_g_mean_mag, gs.phot_bp_mean_mag, gs.phot_rp_mean_mag,
               ap.mh_gspphot, ap.mh_gspspec
        FROM gaiadr3.gaia_source AS gs
        LEFT JOIN gaiadr3.astrophysical_parameters AS ap USING (source_id)
        WHERE gs.source_id IN ({query_ids})
        """

        if verbose:
            stop = min(start + chunk_size, len(unique_ids))
            print(f'Querying Gaia rows {start + 1}-{stop} of {len(unique_ids)} source IDs')

        job = Gaia.launch_job_async(query)
        frame = job.get_results().to_pandas()
        frame['source_id'] = frame['source_id'].astype(str)
        frames.append(frame)

    if not frames:
        return pd.DataFrame(columns=GAIA_COLUMNS)

    gaia_results = pd.concat(frames, ignore_index=True)
    gaia_results = gaia_results.drop_duplicates(subset='source_id').reset_index(drop=True)
    return gaia_results

def get_first_available(row, candidates):
    for column in candidates:
        if column in row.index and pd.notna(row[column]):
            return row[column]
    return pd.NA

def compute_row_kinematics(row):
    ra = get_first_available(row, ['ra_gaia', 'ra', 'ra_y', 'ra_x', 'ra_nea'])
    dec = get_first_available(row, ['dec_gaia', 'dec', 'dec_y', 'dec_x', 'dec_nea'])
    pmra = get_first_available(row, ['pmra'])
    pmdec = get_first_available(row, ['pmdec'])
    radial_velocity = get_first_available(row, ['radial_velocity'])
    parallax = get_first_available(row, ['parallax'])

    required_values = [ra, dec, pmra, pmdec, radial_velocity, parallax]
    if any(pd.isna(value) for value in required_values) or parallax <= 0:
        return pd.Series({column: pd.NA for column in KINEMATIC_COLUMNS})

    U_lsr, V_lsr, W_lsr = UVW(
        ra=ra,
        dec=dec,
        pmra=pmra,
        pmdec=pmdec,
        vrad=radial_velocity,
        plx=parallax,
        warnings=False,
    )
    P_gal = gal_population(U_lsr, V_lsr, W_lsr, char_vel='Mean')[0]
    orbit = compute_orbit_params(
        RA=ra,
        Dec=dec,
        plx=parallax,
        pmRA=pmra,
        pmDec=pmdec,
        Vlos=radial_velocity,
    )

    return pd.Series({
        'U_lsr': U_lsr,
        'V_lsr': V_lsr,
        'W_lsr': W_lsr,
        'P_gal': P_gal,
        'rapo': orbit['rapo'],
        'rperi': orbit['rperi'],
        'eccentricity': orbit['eccentricity'],
        'zmax': orbit['zmax'],
    })

def add_kinematic_columns(df, progress_every=250):
    rows = []
    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        if progress_every and idx % progress_every == 0:
            print(f'Computed kinematics for {idx} rows')
        rows.append(compute_row_kinematics(row))

    kinematics = pd.DataFrame(rows, index=df.index)
    return pd.concat([df, kinematics], axis=1)


In [6]:
gaia_results = fetch_gaia_sources(nea_gaia_clean['source_id'], chunk_size=500)
gaia_results.head()


Querying Gaia rows 1-500 of 4234 source IDs
INFO: Query finished. [astroquery.utils.tap.core]
Querying Gaia rows 501-1000 of 4234 source IDs
INFO: Query finished. [astroquery.utils.tap.core]
Querying Gaia rows 1001-1500 of 4234 source IDs
INFO: Query finished. [astroquery.utils.tap.core]
Querying Gaia rows 1501-2000 of 4234 source IDs
INFO: Query finished. [astroquery.utils.tap.core]
Querying Gaia rows 2001-2500 of 4234 source IDs
INFO: Query finished. [astroquery.utils.tap.core]
Querying Gaia rows 2501-3000 of 4234 source IDs
INFO: Query finished. [astroquery.utils.tap.core]
Querying Gaia rows 3001-3500 of 4234 source IDs
INFO: Query finished. [astroquery.utils.tap.core]
Querying Gaia rows 3501-4000 of 4234 source IDs
INFO: Query finished. [astroquery.utils.tap.core]
Querying Gaia rows 4001-4234 of 4234 source IDs
INFO: Query finished. [astroquery.utils.tap.core]


,source_id,ra,dec,parallax,parallax_error,pmra,pmdec,radial_velocity,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,mh_gspphot,mh_gspspec
0,43802039287826432,57.596227,17.475252,21.471880,0.024102,113.433011,-251.100651,-9.490767,7.348500,7.669333,6.856821,-0.1449,0.13
1,295713106830535424,24.347397,26.952786,26.486350,0.018404,118.236897,-127.589042,-5.739340,10.291925,11.023624,9.455003,0.2110,NaN
2,1695343683312903168,223.092633,70.476557,3.014390,0.012247,-19.045705,-2.479972,-94.878708,11.396728,11.740308,10.887897,-0.0064,0.27
3,2101318556367845888,289.073964,40.366023,2.176931,0.009934,-1.830553,-3.129492,-22.087698,13.323304,13.701756,12.783687,0.1114,NaN
4,2556231154370582400,11.480235,6.347032,8.688432,0.022602,-1.850339,10.445623,-26.530067,12.048571,12.658250,11.305692,0.1781,-0.06


In [7]:
print(f"NEA rows with Gaia IDs: {len(nea_gaia_clean)}")
print(f"Unique Gaia source IDs: {nea_gaia_clean['source_id'].nunique()}")
print(f"Gaia rows returned: {len(gaia_results)}")
print(f"Duplicate Gaia source_id rows: {gaia_results.duplicated(subset='source_id').sum()}")
print(f"Rows with full astrometry for kinematics: {gaia_results[['ra', 'dec', 'pmra', 'pmdec', 'radial_velocity', 'parallax']].notna().all(axis=1).sum()}")


NEA rows with Gaia IDs: 5774
Unique Gaia source IDs: 4234
Gaia rows returned: 4234
Duplicate Gaia source_id rows: 0
Rows with full astrometry for kinematics: 3210


In [8]:
nea_gaia_clean_merged = nea_gaia_clean.merge(
    gaia_results,
    on='source_id',
    how='left',
    validate='m:1',
    suffixes=('_nea', '_gaia'),
)
nea_gaia_with_kinematics = add_kinematic_columns(nea_gaia_clean_merged)
nea_gaia_with_kinematics.to_csv('nea_gaia_full_table.csv', index=False)
nea_gaia_with_kinematics[['source_id'] + KINEMATIC_COLUMNS].head()


Computed kinematics for 250 rows
Computed kinematics for 500 rows
Computed kinematics for 750 rows
Computed kinematics for 1000 rows
Computed kinematics for 1250 rows
Computed kinematics for 1500 rows
Computed kinematics for 1750 rows
Computed kinematics for 2000 rows
/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/PowerSphericalPotentialwCutoff.py:101: RuntimeWarning: overflow encountered in scalar power
  return -self._mass(r) * R / r**3.0

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/MiyamotoNagaiPotential.py:73: RuntimeWarning: overflow encountered in scalar power
  return -R / (R**2.0 + (self._a + numpy.sqrt(z**2.0 + self._b2)) ** 2.0) ** (

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/PowerSphericalPotentialwCutoff.py:218: RuntimeWarning: overflow encountered in power
  -((R[~numpy.isinf(R)] / self.rc) ** 2.0),

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/PowerSphericalPotent

,source_id,U_lsr,V_lsr,W_lsr,P_gal,rapo,rperi,eccentricity,zmax
0,3684934540986187776,4.740412,9.381666,-1.582349,99.2,8.948943,8.098465,0.049889,0.047395
1,105332908999068032,-63.105433,-17.259945,4.685592,97.6,9.559196,6.056471,0.224308,0.069776
2,4678605152248591488,53.802419,-61.441874,-34.696715,47.1,8.636223,4.282972,0.33696,0.640797
3,525352325110310400,19.550779,-3.333874,-5.871674,99.1,8.793972,7.479925,0.080746,0.090918
4,105332908999068032,-63.105433,-17.259945,4.685592,97.6,9.559196,6.056471,0.224308,0.069776


In [13]:
nea_gaia_with_kinematics.to_csv('nea_gaia_full_table.csv', index=False)